# PA3 · Task 3 · Live experiment: one long response vs parallel sampling vs sequential refinement (Step 4)

**What this notebook does.** On the fixed 50 MATH questions (the first 50 held-out questions of the split), with the same 3,072-token output allowance per question:

| Strategy | Generation |
|---|---|
| Single longer response | one response, ≤ 3,072 new tokens |
| Parallel sampling | three independent responses, ≤ 1,024 each; self-consistency picks the answer |
| Sequential refinement | the first parallel response, then one critique-and-revision response (≤ 2,048) |

Model `Qwen/Qwen2.5-3B-Instruct` at the pinned revision from the supplied README, FP16, no quantization, batch size 1, temperature 0.7 / top-p 0.8 / top-k 20 / repetition penalty 1.0, independently seeded calls (seed = 6304 + 10 × question + call), all on **one GPU** so runtimes are comparable. Every call is saved as soon as it finishes.

**Settings:** Accelerator **GPU T4** (one GPU is used), Internet **On** (downloads the model). Roughly 1.5–4 h.
**Attach:** `pa3-task3` and your **`atml-pa3-data`** dataset. If a session stops early, run again with this notebook's earlier output attached as well: finished calls are reused.

**Run with "Save Version → Save & Run All".** **How to read the output:** `PASS / FAIL / OBSERVE`, then the **expected** and **actual** lines.

In [1]:
# S1. Find the Task 3 code (attached Kaggle dataset, or a clone of your GitHub repo) and copy it to /tmp.
import os, sys, glob, json, time, shutil, subprocess
from pathlib import Path
SESSION_START = time.time()     # Kaggle's 12-hour limit counts from about now

REPO_URL = ""        # optional: your public GitHub repo URL, used only if no code dataset is attached
REQUIRED_CODE_VERSION = "2026.11.03"   # this notebook needs this code version or newer
NEED_GPU = True
CODE = Path("/tmp/pa3_task3_code")

def code_version(root):
    import re
    try:
        m = re.search(r'CODE_VERSION\s*=\s*"([^"]+)"', (Path(root) / "common" / "__init__.py").read_text())
        return m.group(1) if m else "0 (no version stamp)"
    except FileNotFoundError:
        return "0 (no version stamp)"

hits = sorted({Path(p).parents[1] for p in glob.glob("/kaggle/input/**/task3/t3_offline.py", recursive=True)},
              key=code_version, reverse=True)
for h in hits:
    print(f"found code copy: {h}  (version {code_version(h)})")
if CODE.exists():
    shutil.rmtree(CODE)
if hits:
    if code_version(hits[0]) < REQUIRED_CODE_VERSION:
        raise SystemExit(f"The newest attached code is version {code_version(hits[0])}, but this notebook needs "
                         f"{REQUIRED_CODE_VERSION}. Upload the latest pa3-task3.zip as a new dataset version and re-attach it.")
    shutil.copytree(hits[0], CODE)
    print("using:", hits[0])
elif REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, str(CODE)], check=True)
    print("code cloned from", REPO_URL)
else:
    raise SystemExit("Attach the pa3-task3 dataset (or set REPO_URL) and re-run this cell.")
sys.path.insert(0, str(CODE))
os.chdir(CODE)
CODE_COPIES = [str(h) for h in hits]
def inputs(pattern):
    """Files matching `pattern` in the ATTACHED inputs only (never inside a copy of the code)."""
    return [p for p in glob.glob(pattern, recursive=True) if not any(p.startswith(c + "/") for c in CODE_COPIES)]
from common.utils import Checks, human_bytes, dir_size_bytes, read_json, read_jsonl
CHK = Checks()
CHK.check(code_version(CODE) >= REQUIRED_CODE_VERSION, "code version in use", f">= {REQUIRED_CODE_VERSION}", code_version(CODE))
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
    GPUS = [l.strip() for l in smi.stdout.strip().splitlines() if l.strip()]
except FileNotFoundError:
    GPUS = []
if NEED_GPU:
    CHK.check(len(GPUS) >= 1, "GPU visible", "GPU T4 (Settings -> Accelerator)", GPUS or "no GPU: is the accelerator switched on?")
else:
    CHK.observe("accelerator", GPUS or "CPU only (enough for this notebook)")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "math-verify==0.9.0"], check=True)
import importlib.metadata as md
vers = {p: md.version(p) for p in ["torch", "transformers", "numpy", "pandas", "pyarrow", "scipy", "math-verify"]}
CHK.observe("library versions", vers)
CHK.check(vers["math-verify"] == "0.9.0", "answer-equivalence scorer pinned (math-verify)", "0.9.0", vers["math-verify"])
# the supplied rollouts, attached as a Kaggle dataset (ATML-PA3-Data, unchanged)
from task3 import t3_data as D
FOLDERS = D.find_folders(["/kaggle/input"])
CHK.check(set(FOLDERS) == {"math", "mathvista"}, "supplied data attached (qwen25-math/, qwen25vl-mathvista/)", "both folders",
          {k: str(v) for k, v in FOLDERS.items()} or "not found: attach your atml-pa3-data dataset")
DATA_ROOT = str(Path(FOLDERS["math"]).parent) if "math" in FOLDERS else "/kaggle/input"
for k, v in FOLDERS.items():
    CHK.observe(f"{k} files", {f.name: human_bytes(f.stat().st_size) for f in sorted(Path(v).glob("*.parquet"))})
WORK = Path("/kaggle/working"); T3 = WORK / "t3"; T3.mkdir(parents=True, exist_ok=True)

found code copy: /kaggle/input/datasets/zylah628/pa3-task3/pa3-task3  (version 2026.11.03)
using: /kaggle/input/datasets/zylah628/pa3-task3/pa3-task3
[PASS   ] code version in use
          expected: >= 2026.11.03
          actual  : 2026.11.03
[PASS   ] GPU visible
          expected: GPU T4 (Settings -> Accelerator)
          actual  : ['Tesla T4, 15360 MiB', 'Tesla T4, 15360 MiB']
[OBSERVE] library versions
          actual  : {'torch': '2.11.0+cu130', 'transformers': '5.20.2', 'numpy': '2.1.3', 'pandas': '2.3.3', 'pyarrow': '23.0.1', 'scipy': '1.16.3', 'math-verify': '0.9.0'}
[PASS   ] answer-equivalence scorer pinned (math-verify)
          expected: 0.9.0
          actual  : 0.9.0
[PASS   ] supplied data attached (qwen25-math/, qwen25vl-mathvista/)
          expected: both folders
          actual  : {'math': '/kaggle/input/datasets/zylah628/atml-pa3-data/ATML-PA3-Data/qwen25-math', 'mathvista': '/kaggle/input/datasets/zylah628/atml-pa3-data/ATML-PA3-Data/qwen25vl-mathvista'}
[OB

In [2]:
# S2. Launch generation on GPU 0 (separate process); earlier calls from an attached output are reused.
STOP_AFTER_HOURS = 11.0
prev = sorted(inputs("/kaggle/input/**/t3/live/live_calls.jsonl"))
for p in prev: print("reusing earlier calls from", p, f"({sum(1 for _ in open(p))} calls)")
LOG = WORK / "t3_live.log"
cmd = [sys.executable, "-u", "-m", "task3.t3_live", "generate", "--data_root", DATA_ROOT, "--out", str(T3),
       "--stop_at", str(SESSION_START + STOP_AFTER_HOURS * 3600)] + (["--resume_from", *prev] if prev else [])
proc = subprocess.Popen(cmd, cwd=CODE, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})
print("started:", " ".join(cmd[3:5]))

started: task3.t3_live generate


In [3]:
# S3. Monitor: prints progress every 10 minutes until generation ends.
def tail(n=2):
    try: return "".join([l for l in open(LOG) if l.strip() and "it/s]" not in l and "%|" not in l][-n:]).strip()
    except FileNotFoundError: return "(not started)"
t0, tp = time.time(), time.time()
while proc.poll() is None:
    time.sleep(30)
    if time.time() - tp >= 600:
        tp = time.time(); print(f"--- {(time.time() - t0) / 60:.0f} min: {tail()}")
print(tail(40))
CHK.check(proc.returncode == 0, "generation process exited cleanly", 0, proc.returncode)
PAUSED = (T3 / "live" / "PAUSED").exists()
if PAUSED: print("PAUSED at the time limit: run this notebook again with THIS version's output attached to finish.")

--- 10 min: 2 / 50 questions finished (4.0 min this session)
  3 / 50 questions finished (8.7 min this session)
--- 20 min: 5 / 50 questions finished (13.8 min this session)
  6 / 50 questions finished (16.3 min this session)
--- 30 min: 9 / 50 questions finished (24.0 min this session)
  10 / 50 questions finished (28.1 min this session)
--- 40 min: 13 / 50 questions finished (35.0 min this session)
  14 / 50 questions finished (38.3 min this session)
--- 50 min: 17 / 50 questions finished (47.1 min this session)
  18 / 50 questions finished (48.5 min this session)
--- 60 min: 19 / 50 questions finished (55.2 min this session)
  20 / 50 questions finished (57.1 min this session)
--- 70 min: 24 / 50 questions finished (66.8 min this session)
  25 / 50 questions finished (68.2 min this session)
--- 80 min: 28 / 50 questions finished (73.6 min this session)
  29 / 50 questions finished (78.1 min this session)
--- 90 min: 32 / 50 questions finished (83.3 min this session)
  33 / 50 questi

In [4]:
# S4. Score the three strategies: accuracy, actual tokens, runtime, truncation / extraction, revision transitions.
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
r = subprocess.run([sys.executable, "-u", "-m", "task3.t3_live", "score", "--out", str(T3)], cwd=CODE, capture_output=True, text=True)
print(r.stdout[-6000:]); print(r.stderr[-3000:] if r.returncode else "")
LIVE_OK = r.returncode == 0 and "0 FAIL" in r.stdout
CHK.check(LIVE_OK, "all 50 questions scored", "0 FAIL", [l for l in r.stdout.splitlines() if l.startswith("====")][-1:] or r.returncode)
if LIVE_OK:
    print(pd.read_csv(T3 / "live" / "live_summary.csv").round(3).T.to_string())
    print(pd.read_csv(T3 / "live" / "live_bootstrap.csv").round(2).to_string(index=False))
    print(json.dumps(read_json(T3 / "live" / "live_transitions.json"), indent=1))
    CHK.observe("environment", read_json(T3 / "live" / "live_env.json"))

[PASS   ] every live question has all five calls
          expected: 50
          actual  : 50
[OBSERVE] live: strategies (50 MATH questions)
          actual  : [{'strategy': 'single longer response (3,072)', 'accuracy': 0.64, 'questions': 50, 'mean input tokens': 108.38, 'mean output tokens': 642.26, 'unused output allowance (tokens)': 2429.74, 'total runtime (min)': 26.064, 'mean runtime per question (s)': 31.276, 'truncated calls (%)': 0.0, 'calls without an extracted answer (%)': 0.0, 'final answer missing (%)': 0.0, 'peak GPU memory (GB)': 6.532}, {'strategy': 'parallel x3 (1,024) + self-consistency', 'accuracy': 0.68, 'questions': 50, 'mean input tokens': 325.14, 'mean output tokens': 1843.1, 'unused output allowance (tokens)': 1228.9, 'total runtime (min)': 74.3, 'mean runtime per question (s)': 89.161, 'truncated calls (%)': 18.0, 'calls without an extracted answer (%)': 16.0, 'final answer missing (%)': 16.0, 'peak GPU memory (GB)': 6.376}, {'strategy': 'initial (1,024) + cri

In [5]:
# S5. Illustrative refinement cases (stated rule: the first questions, in split order, where the revision changed the answer).
if LIVE_OK:
    pq = pd.read_csv(T3 / "live" / "live_per_question.csv")
    ch = pq[pq.revision_changed_answer].head(4)
    calls = {(c["sample_id"], c["call"]): c for c in read_jsonl(T3 / "live" / "live_calls.jsonl")}
    for _, x in ch.iterrows():
        a, b = calls[(x.sample_id, "parallel_1")], calls[(x.sample_id, "revision")]
        print(f"\n######## {x.sample_id}  gold {x.gold}  initial {x.initial_answer} ({'right' if x.initial_correct else 'wrong'}) -> "
              f"revised {x.revised_answer} ({'right' if x.revised_correct else 'wrong'})")
        print("INITIAL (end):", a["response"][-500:].replace("\n", " "))
        print("REVISION (start):", b["response"][:700].replace("\n", " "))


######## test:3504  gold 216  initial nan (wrong) -> revised 2520 (wrong)
INITIAL (end): another prime factor \( p_3 \) such that the total number of factors is 16. Let's try \( 16 = 4 \times 4 \) with additional primes.    - \( 16 = 4 \times 4 \) implies \( e_1 = 3 \) and \( e_2 = 3 \). We need to include another prime factor \( p_3 \) with \( e_3 = 1 \).  8. **Construct the number:**    - The smallest primes are 2, 3, and 5. Let's use \( 2^3 \times 3^3 \times 5^1 \):    \[    n = 2^3 \times 3^3 \times 5^1 = 8 \times 27 \times 5 = 1080    \]  9. **Verify the number of factors:**   
REVISION (start): Let's review the solution step-by-step to ensure all steps are correct and identify any necessary corrections.  1. **Factorize 72:**    \[    72 = 2^3 \times 3^2    \]  2. **Understand the number of factors:**    The number of factors of \( n = p_1^{e_1} \times p_2^{e_2} \times \cdots \times p_k^{e_k} \) is given by:    \[    (e_1 + 1)(e_2 + 1) \cdots (e_k + 1)    \]    We need this produ

In [6]:
# S6. Machine-readable results for the GitHub repo: t3_live_results.zip.
import zipfile
with zipfile.ZipFile(WORK / "t3_live_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted((T3 / "live").rglob("*")):
        if f.is_file(): z.write(f, str(f.relative_to(WORK)))
CHK.observe("t3_live_results.zip", human_bytes((WORK / "t3_live_results.zip").stat().st_size))
CHK.summary(); CHK.save(WORK / "t3_live_notebook_checks.json")
print("\nNEXT:", "send this notebook (S3-S5 output) and t3_live_results.zip." if LIVE_OK else
      "not finished: run again with pa3-task3, atml-pa3-data AND this version's output attached.")

[OBSERVE] t3_live_results.zip
          actual  : 94.65 KB

==== 6 PASS | 0 FAIL | 5 OBSERVE ====

NEXT: send this notebook (S3-S5 output) and t3_live_results.zip.
